In [1]:
import pandas as pd

# Load the training and test datasets
train_df = pd.read_csv('D:\\LLM-Driven_AI-Studio\\MLAgent\\data\\benchmark/DSEval/datasets/04_diabetes/train.csv')
test_df = pd.read_csv('D:\\LLM-Driven_AI-Studio\\MLAgent\\data\\benchmark/DSEval/datasets/04_diabetes/test.csv')

# Drop duplicate rows from the training and test datasets
train_df = train_df.drop_duplicates()
test_df = test_df.drop_duplicates()


In [2]:
from metagpt.tools.libs.data_preprocess import get_column_info

column_info = get_column_info(train_df)
print("column_info")
print(column_info)


2025-09-10 08:04:12.438 | INFO     | metagpt.const:get_metagpt_package_root:29 - Package root set to D:\LLM-Driven_AI-Studio\MLAgent\experiments\DataInterpreter


column_info
{'Category': ['gender', 'smoking_history'], 'Numeric': ['age', 'hypertension', 'heart_disease', 'bmi', 'HbA1c_level', 'blood_glucose_level', 'diabetes'], 'Datetime': [], 'Others': []}


In [3]:
from metagpt.tools.libs.data_preprocess import OneHotEncode

# Initialize the OneHotEncode tool with the specified categorical columns
encoder = OneHotEncode(features=['gender', 'smoking_history'])

# Fit and transform the training data
train_df_encoded = encoder.fit_transform(train_df.copy())

# Transform the test data using the same encoder
test_df_encoded = encoder.transform(test_df.copy())


D:\LLM-Driven_AI-Studio\MLAgent\experiments\DataInterpreter\venv\lib\site-packages\sklearn\preprocessing\_encoders.py:975: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


In [4]:
from metagpt.tools.libs.data_preprocess import get_column_info

# Check column information for the encoded train DataFrame
column_info_train = get_column_info(train_df_encoded)
print("Column information for train_df_encoded:")
print(column_info_train)

# Check column information for the encoded test DataFrame
column_info_test = get_column_info(test_df_encoded)
print("Column information for test_df_encoded:")
print(column_info_test)


Column information for train_df_encoded:
{'Category': [], 'Numeric': ['age', 'hypertension', 'heart_disease', 'bmi', 'HbA1c_level', 'blood_glucose_level', 'diabetes', 'gender_Female', 'gender_Male', 'gender_Other', 'smoking_history_No Info', 'smoking_history_current', 'smoking_history_ever', 'smoking_history_former', 'smoking_history_never', 'smoking_history_not current'], 'Datetime': [], 'Others': []}
Column information for test_df_encoded:
{'Category': [], 'Numeric': ['age', 'hypertension', 'heart_disease', 'bmi', 'HbA1c_level', 'blood_glucose_level', 'diabetes', 'gender_Female', 'gender_Male', 'gender_Other', 'smoking_history_No Info', 'smoking_history_current', 'smoking_history_ever', 'smoking_history_former', 'smoking_history_never', 'smoking_history_not current'], 'Datetime': [], 'Others': []}


In [5]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel
import matplotlib.pyplot as plt
import seaborn as sns

# Assuming 'diabetes' is the target column
X_train = train_df_encoded.drop(columns=['diabetes'])
y_train = train_df_encoded['diabetes']
X_test = test_df_encoded.drop(columns=['diabetes'])
y_test = test_df_encoded['diabetes']

# Build a random forest classifier
rf_classifier = RandomForestClassifier(random_state=42)
rf_classifier.fit(X_train, y_train)

# Rank the most important features
feature_importances = rf_classifier.feature_importances_
feature_names = X_train.columns

# Create a DataFrame to store feature names and their importances
feature_importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': feature_importances
})

# Sort the DataFrame by importance in descending order
feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False)

# Display the feature importances
print(feature_importance_df)

# Plot the feature importances
plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=feature_importance_df)
plt.title('Feature Importances')
plt.show()


                        Feature  Importance
4                   HbA1c_level    0.407970
5           blood_glucose_level    0.312457
3                           bmi    0.125713
0                           age    0.105406
1                  hypertension    0.015265
2                 heart_disease    0.010795
9       smoking_history_No Info    0.003989
12       smoking_history_former    0.003733
13        smoking_history_never    0.003394
10      smoking_history_current    0.002612
7                   gender_Male    0.002222
6                 gender_Female    0.002198
14  smoking_history_not current    0.002194
11         smoking_history_ever    0.002050
8                  gender_Other    0.000002


In [6]:
from metagpt.tools.libs.data_preprocess import get_column_info

# Check column information for the encoded training data
column_info_train = get_column_info(train_df_encoded)
print("Column Information for Training Data")
print(column_info_train)

# Check column information for the encoded test data
column_info_test = get_column_info(test_df_encoded)
print("Column Information for Test Data")
print(column_info_test)


Column Information for Training Data
{'Category': [], 'Numeric': ['age', 'hypertension', 'heart_disease', 'bmi', 'HbA1c_level', 'blood_glucose_level', 'diabetes', 'gender_Female', 'gender_Male', 'gender_Other', 'smoking_history_No Info', 'smoking_history_current', 'smoking_history_ever', 'smoking_history_former', 'smoking_history_never', 'smoking_history_not current'], 'Datetime': [], 'Others': []}
Column Information for Test Data
{'Category': [], 'Numeric': ['age', 'hypertension', 'heart_disease', 'bmi', 'HbA1c_level', 'blood_glucose_level', 'diabetes', 'gender_Female', 'gender_Male', 'gender_Other', 'smoking_history_No Info', 'smoking_history_current', 'smoking_history_ever', 'smoking_history_former', 'smoking_history_never', 'smoking_history_not current'], 'Datetime': [], 'Others': []}


In [7]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, confusion_matrix
import numpy as np

# Initialize the Random Forest Classifier with suitable hyperparameters
rf_classifier = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42
)

# Fit the model on the training set
rf_classifier.fit(X_train, y_train)

# Make predictions on the test set
y_pred = rf_classifier.predict(X_test)
y_pred_proba = rf_classifier.predict_proba(X_test)[:, 1]

# Compute the area under the ROC curve
roc_auc = roc_auc_score(y_test, y_pred_proba)
print(f'Area Under ROC Curve: {roc_auc:.4f}')

# Show the confusion matrix
conf_matrix = confusion_matrix(y_test, y_pred)
print('Confusion Matrix:')
print(conf_matrix)


Area Under ROC Curve: 0.9704
Confusion Matrix:
[[18099     0]
 [  556  1152]]


In [8]:
# Make predictions on the test set using the trained model
y_pred = rf_classifier.predict(X_test)
y_pred_proba = rf_classifier.predict_proba(X_test)[:, 1]

# Display the predictions and probabilities
print("Predictions:", y_pred)
print("Probabilities:", y_pred_proba)


Predictions: [0 0 0 ... 0 0 0]
Probabilities: [0.00667895 0.00197012 0.02538687 ... 0.00291565 0.06126535 0.02303555]


In [9]:
from sklearn.metrics import roc_auc_score

# Ensure the test data is processed the same way as the training data
X_test = test_df_encoded.drop(columns=['diabetes'])
y_test = test_df_encoded['diabetes']

# Use the trained model to predict probabilities
y_pred_proba = rf_classifier.predict_proba(X_test)[:, 1]

# Compute the area under the ROC curve
roc_auc = roc_auc_score(y_test, y_pred_proba)
print(f'Area Under ROC Curve: {roc_auc:.4f}')


Area Under ROC Curve: 0.9704


In [10]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Assuming rf_classifier and test_df_encoded are already defined from previous tasks
X_test = test_df_encoded.drop(columns=['diabetes'])
y_test = test_df_encoded['diabetes']
y_pred = rf_classifier.predict(X_test)

# Compute confusion matrix
conf_matrix = confusion_matrix(y_test, y_pred)

# Plot confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', xticklabels=['No Diabetes', 'Diabetes'], yticklabels=['No Diabetes', 'Diabetes'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()
